## Week-2

In [2]:
import pandas as pd

df = pd.read_csv("CRMLSSold_residential.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_3348/1797885786.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSSold_residential.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,94401,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
1,SanDiego,SanDiego,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,davidmartz@compass.com,...,92262,NaN,13504.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
4,Southland,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,karen.klein@theagencyre.com,...,91356,0.0,17873.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN


In [3]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns
print("Categorical columns:", len(cat_cols))
print(cat_cols)

Categorical columns: 52
Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'ListAgentEmail', 'CloseDate',
       'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress',
       'PropertyType', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish',
       'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'BuilderName',
       'PropertySubType', 'SubdivisionName', 'BuyerOfficeAOR', 'ListingId',
       'City', 'ContractStatusChangeDate', 'CoBuyerAgentFirstName',
       'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince',
       'MiddleOrJuniorSchool', 'FireplaceYN', 'HighSchool', 'Levels',
       'LotSizeDimensions', 'NewConstructionYN', 'HighSchoolDistrict',
       'PostalCode', 'Originating

In [4]:
import pandas as pd

# =========================
# 1. Identify categorical columns
# =========================
cat_cols = df.select_dtypes(include=["object", "string"]).columns

# =========================
# 2. Build base review table
# =========================
cat_review = pd.DataFrame({
    "column": cat_cols,
    "missing_count": df[cat_cols].isnull().sum().values,
    "missing_pct": (df[cat_cols].isnull().mean().values * 100).round(2),
    "n_unique": df[cat_cols].nunique(dropna=True).values
})

# =========================
# 3. Sample top values (for quick inspection)
# =========================
top_values = []
for col in cat_cols:
    vals = df[col].value_counts(dropna=False).head(5)
    vals_str = "; ".join([f"{str(idx)} ({cnt})" for idx, cnt in vals.items()])
    top_values.append(vals_str)

cat_review["top_values_sample"] = top_values

# =========================
# 4. Define core categorical columns to keep
# =========================
core_columns = [
    "PropertyType",
    "PropertySubType",
    "City",
    "PostalCode",
    "CountyOrParish",
    "StateOrProvince",
    "MLSAreaMajor",
    "MlsStatus",
    "CloseDate",
    "ContractStatusChangeDate",
    "PurchaseContractDate"
]

# =========================
# 5. Decision logic
# =========================
def classify_column(row):
    col = row["column"]
    missing_pct = row["missing_pct"]
    
    if col in core_columns:
        return "Keep (Core)"
    elif missing_pct >= 90:
        return "Drop"
    else:
        return "Review"


cat_review["decision"] = cat_review.apply(classify_column, axis=1)
# =========================
# 6. Sort by missing percentage
# =========================
cat_review = cat_review.sort_values(by="missing_pct", ascending=False)

# =========================
# 7. Display results
# =========================
print("=== KEEP (CORE) ===")
display(cat_review[cat_review["decision"] == "Keep (Core)"])

print("=== DROP ===")
display(cat_review[cat_review["decision"] == "Drop"])

print("=== Review ===")
display(cat_review[cat_review["decision"] == "Review"])



=== KEEP (CORE) ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
23,MLSAreaMajor,53122,13.36,1088,nan (53122); 699 - Not Defined (41925); SRCAR ...,Keep (Core)
29,PropertySubType,776,0.20,20,SingleFamilyResidence (297332); Condominium (6...,Keep (Core)
34,ContractStatusChangeDate,589,0.15,818,2024-06-28 (1434); 2024-03-29 (1326); 2026-02-...,Keep (Core)
33,City,310,0.08,1123,San Diego (17870); Los Angeles (16248); San Jo...,Keep (Core)
36,PurchaseContractDate,194,0.05,1135,2024-04-24 (874); 2024-05-01 (848); 2024-05-22...,Keep (Core)
46,PostalCode,2,0.00,3582,92253 (2566); 92211 (2120); 92262 (1799); 9222...,Keep (Core)
12,PropertyType,0,0.00,1,Residential (397603),Keep (Core)
8,CloseDate,0,0.00,818,2024-06-28 (1434); 2024-03-29 (1326); 2026-02-...,Keep (Core)
38,StateOrProvince,0,0.00,15,CA (397575); AZ (9); OS (4); NV (2); CO (2),Keep (Core)
25,MlsStatus,0,0.00,1,Closed (397603),Keep (Core)


=== DROP ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
4,WaterfrontYN,397355,99.94,1,nan (397355); True (248),Drop
5,BasementYN,389826,98.04,1,nan (389826); True (7777),Drop
43,LotSizeDimensions,378282,95.14,9656,nan (378282); 50x135 (335); 50x100 (243); 50x1...,Drop
28,BuilderName,378056,95.08,3284,nan (378056); Lennar (1822); Richmond American...,Drop
35,CoBuyerAgentFirstName,361576,90.94,4311,nan (361576); Michael (527); Andrew (445); Dav...,Drop
48,OriginatingSystemSubName,358351,90.13,10,nan (358351); CRMLS_CRM (21660); CRMLS_SAND (2...,Drop
47,OriginatingSystemName,358351,90.13,1,nan (358351); CRMLS (39252),Drop


=== Review ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
49,BuyerAgencyCompensationType,351467,88.40,3,nan (351467); Item1 (45479); Item (613); SeeRe...,Review
26,ElementarySchool,344627,86.68,1394,nan (344627); Other (9117); Harbor View (324);...,Review
39,MiddleOrJuniorSchool,344273,86.59,660,nan (344273); Other (6783); Rancho Santa Marga...,Review
50,latfilled,333719,83.93,2,nan (333719); True (33427); False (30457),Review
51,lonfilled,333719,83.93,2,nan (333719); True (33427); False (30457),Review
41,HighSchool,328353,82.58,520,nan (328353); Other (2721); Tesoro (1117); San...,Review
17,CoListAgentFirstName,306611,77.11,6568,nan (306611); Michael (1334); David (1198); Ka...,Review
18,CoListAgentLastName,306376,77.06,13641,nan (306376); Myatt (894); Smith (532); Flores...,Review
15,CoListOfficeName,303521,76.34,5433,nan (303521); Compass (10171); Coldwell Banker...,Review
30,SubdivisionName,249875,62.85,15782,nan (249875); Other (4048); Not Listed (2322);...,Review


##Second Round of Screening: Review Phase

Drop: OriginatingSystemSubName, OriginatingSystemName,
    CoListAgentFirstName, CoListAgentLastName, BuyerAgentAOR,
    ListAgentAOR, ListAgentEmail, BuyerOfficeAOR, ListAgentFirstName, 
    BuyerAgentFirstName, BuyerAgentLastName, ListAgentFullName

keep: 
ElementarySchool: -School attendance information for the assigned elementary school.

MiddleOrJuniorSchool: -School attendance information for the assigned middle or junior school.

BuyerAgencyCompensationType:- Type of compensation offered to the buyer’s agent.

HighSchool: -School attendance information for the assigned high school.

latfilled: -Filled or imputed latitude value for the property location.

lonfilled: -Filled or imputed longitude value for the property location.

AssociationFeeFrequency: -Frequency of HOA fee payments, such as monthly or yearly.

Flooring: -List or description of flooring materials used in the property.

HighSchoolDistrict: -High school district information associated with the property.

AttachedGarageYN: -Indicates whether the property has an attached garage (Yes/No).

Levels: -Number of levels or floors in the property.

PoolPrivateYN: -Indicates whether the property has a private pool.

NewConstructionYN: -Indicates whether the property is newly constructed.

ViewYN: -Indicates whether the property has a notable view (e.g., ocean, city).

FireplaceYN: -Indicates whether the property has a fireplace.


BuyerOfficeName: Name of the buyer’s brokerage office.

BuyerAgentMlsId: Internal MLS identifier for the buyer’s agent.

UnparsedAddress: Full unstructured property address.

ListingContractDate: Date when the property was listed on the market.

ListOfficeName: Name of the listing (seller’s) brokerage office.

ListingId: Unique identifier for the property listing, used for tracking or display.
